# ⚾ 재대결에서 "구종을 바꾸는 선택" 자체의 가치 평가

* **작성자:** DY
* **작성일:** 2026-10-05

## 1. 배경: 이 프로젝트의 질문, 그리고 이전 시도가 왜 답을 못 했는가

이 프로젝트는 투수가 장타(2루타 이상)를 맞은 뒤 같은 타자와 다시 만나는 **재대결** 타석에서, 그때 맞은
구종을 피하는 경향(**회피 성향**)이 있는지, 그리고 그 회피가 실제로 투수에게 유리한 선택인지를 봅니다.

이전 노트북(03, 07)은 **RVAE**(실제 결과 − 예측된 기대 결과)라는 지표로 이 질문에 답하려 했습니다.
결정구마다 "이 상황(볼카운트·주자·점수차 등)에서 **그때 실제로 던진 구종**을 던지면 평균적으로 얼마나
피해를 주는가"를 예측하는 GBM(gradient boosting) 모델을 만들고, 실제 결과에서 그 예측값을 뺀 값이
RVAE입니다.

문제는 GBM의 예측 피처에 **실제로 던진 구종(`pitch_type`)**이 그대로 들어가 있다는 점입니다. 그래서
"이 구종을 고른 것 자체의 이득"이 이미 예측값 쪽에 녹아들어가 있고, 실제 결과에서 그 예측값을 빼면 그
이득까지 같이 지워집니다. 예를 들어 회피가 실제로 유리해서 회피한 쪽의 평균 피해가 더 낮다고 해도,
GBM이 "그 구종이면 원래 이 정도 피해"라고 정확히 학습했다면 RVAE는 양쪽 다 0에 가깝게 나옵니다 —
"선택이 좋았는가"가 아니라 "선택을 하고 난 뒤 실행이 예상만큼이었는가"만 남는 구조적 문제입니다.
실제로 노트북 07에서 이 방식으로 돌린 혼합효과 모델은 전부 비유의한 결과가 나왔습니다.

이 노트북은 같은 질문을 **이 문제가 구조적으로 없는 다른 지표**로 다시 검증합니다.

## 2. 접근 방법: 선택가치 (counterfactual swap)

"그 구종을 고른 것 자체가 좋은 선택이었는가"를 직접 재려면, 실제 결과와 예측값을 비교하는 대신
**같은 상황에서 다른 구종을 던졌다면 어땠을지와 비교**해야 합니다. 그래서 노트북 03의 GBM(상황 +
실제로 던진 구종 → 기대 피해를 예측하는 모델)을 그대로 쓰되, 같은 상황에서 `pitch_type`만 "실제로
던진 구종" → "**맞은 구종(재사용했다고 가정)**"으로 바꿔서 두 번 예측합니다.

$$\text{선택가치} = \text{예측(맞은 구종을 재사용했다고 가정)} - \text{예측(실제로 던진 구종)}$$

양수면 실제 선택이 "맞은 구종을 또 던지는 것"보다 더 나았다는 뜻입니다. 이건 **실제 결과(`woba_value`)를
전혀 쓰지 않는** 순수 모델-내적 비교라, RVAE의 상쇄 문제가 없습니다 — GBM이 학습한 "구종별 기대 피해"
관계만으로 계산되기 때문입니다.

회귀의 공변량에서는 `balls`/`strikes`(결정구 시점 값이라 타석이 어떻게 진행됐는지의 결과물,
post-treatment)와 `pitch_family`(선택가치의 정의 자체와 순환)를 **의도적으로 제외**합니다 —
`outs_when_up`/`score_diff`/`platoon_match`/`risp`/`runners_n`/`season`만 통제합니다(이들은 타석이
어떻게 진행됐든 거의 바뀌지 않는, 타석 시작 이전 상태에 가깝습니다 — 8절에서 이 가정 자체를 실측으로
검증합니다).

In [1]:
# 1. 경로 설정 (src 폴더 접근용)
import sys, os
sys.path.append(os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score

from src.analysis.avoidance_stats import compute_season_usage_rate
from src.analysis.glmer_runner import (
    check_convergence,
    extract_lmer_fixed_effects,
    extract_variance_components,
    fit_lmer,
)
from src.analysis.mixed_effects_model import compute_icc_gaussian
from src.analysis.run_same_batter_rematch_analysis import build_xbh_rematch
from src.preprocessing.pitch_family import map_pitch_family

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)

TRAIN_SEASONS = (2021, 2022, 2023, 2024, 2025)
VALID_SEASON = 2026
RANDOM_STATE = 20261003

Error importing in API mode: ImportError("dlopen(/Users/dongyunkwak/GitHub/9th-first-project-baseball-2/.venv/lib/python3.14/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): Library not loaded: /Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.dylib\n  Referenced from: <9F0E20C3-D782-31B5-943E-66F62BCE77E8> /Users/dongyunkwak/GitHub/9th-first-project-baseball-2/.venv/lib/python3.14/site-packages/_rinterface_cffi_api.abi3.so\n  Reason: tried: '/Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.dylib' (no such file), '/Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.dylib' (no such file)")


Trying to import in ABI mode.


## 3. 데이터 준비

팀 CSV를 읽고, 볼카운트·주자·점수차 같은 상황 피처를 계산하고, 장타 후 같은 타자와의 재대결 이벤트를
추출하고, "상황 + 구종 → 기대 피해(`woba_value`)"를 예측하는 GBM을 학습하고, 마지막으로 각 재대결
타석이 어떤 결정구(그 타석을 끝낸 투구)로 끝났는지 매칭합니다. 노트북 07과 완전히 같은 과정입니다.

In [2]:
from src.preprocessing.research_sample import KEY_COLUMNS, list_research_csvs
from src.preprocessing.build_next_ab_dataset import EXTRA_BASE_HIT_EVENTS

LOAD_COLUMNS = [
    'game_pk', 'game_date', 'at_bat_number', 'pitch_number', 'pitcher', 'player_name', 'batter',
    'stand', 'p_throws', 'pitch_type', 'events', 'balls', 'strikes', 'outs_when_up',
    'inning', 'inning_topbot', 'on_1b', 'on_2b', 'on_3b', 'home_score', 'away_score',
    'woba_value', 'woba_denom',
]


def load_team_csv() -> pd.DataFrame:
    frames = [
        pd.read_csv(path, usecols=LOAD_COLUMNS, encoding='utf-8-sig', low_memory=False)
        for path in list_research_csvs()
    ]
    pitches = pd.concat(frames, ignore_index=True)
    pitches['season'] = pd.to_datetime(pitches['game_date']).dt.year
    return pitches


pitches = load_team_csv()
assert len(pitches) == 3_592_302 and pitches['pitcher'].nunique() == 1_067
assert not pitches.duplicated(KEY_COLUMNS).any()
print(f'{len(pitches):,}행, 투수 {pitches["pitcher"].nunique():,}명')

3,592,302행, 투수 1,067명


In [3]:
SITUATION_COLUMNS = ['runners_n', 'risp', 'score_diff', 'platoon_match']


def add_situation_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    on1, on2, on3 = out['on_1b'].notna(), out['on_2b'].notna(), out['on_3b'].notna()
    out['runners_n'] = on1.astype(int) + on2.astype(int) + on3.astype(int)
    out['risp'] = (on2 | on3).astype(int)
    out['score_diff'] = np.where(
        out['inning_topbot'] == 'Top', out['home_score'] - out['away_score'], out['away_score'] - out['home_score']
    )
    out['platoon_match'] = np.where(
        out['stand'].isna() | out['p_throws'].isna(), np.nan, (out['stand'] == out['p_throws']).astype(float)
    )
    return out


_t = pd.DataFrame({
    'on_1b': [np.nan, 1.0, np.nan], 'on_2b': [np.nan, np.nan, 2.0], 'on_3b': [np.nan, np.nan, np.nan],
    'inning_topbot': ['Top', 'Bot', 'Top'], 'home_score': [3, 5, 2], 'away_score': [3, 2, 4],
    'stand': ['L', 'R', 'R'], 'p_throws': ['R', 'R', 'L'],
})
_r = add_situation_columns(_t)
assert _r['runners_n'].tolist() == [0, 1, 1]
assert _r['risp'].tolist() == [0, 0, 1]
assert _r['score_diff'].tolist() == [0, -3, -2]
assert _r['platoon_match'].tolist() == [0.0, 1.0, 0.0]
print('add_situation_columns 테스트 통과')

pitches = add_situation_columns(pitches)

add_situation_columns 테스트 통과


In [4]:
season_usage = compute_season_usage_rate(pitches)
xbh = build_xbh_rematch(pitches, season_usage)
xbh = xbh[xbh['has_next_ab']].copy()
assert len(xbh) == 27_391, len(xbh)
print(f'재대결 이벤트 {len(xbh):,}건')

2026-10-05 22:48:58,881 [INFO] 장타 70319건 중 같은 타자·같은 투수 재대결 27391건 (39.0%)
        xbh_total  same_batter_rematch  rematch_share
season                                               
2021        12709                 4688         0.3689
2022        12186                 4623         0.3794
2023        13137                 5280         0.4019
2024        12427                 4977         0.4005
2025        12394                 4916         0.3966
2026         7466                 2907         0.3894


재대결 이벤트 27,391건


In [5]:
MODEL_FEATURES = [
    'pitch_type', 'stand', 'p_throws', 'balls', 'strikes', 'outs_when_up', 'runners_n', 'risp', 'score_diff',
    'season_usage_rate',
]
CATEGORICAL_FEATURES = ['pitch_type', 'stand', 'p_throws']


def build_decisive_pitch_table(pitches_sit: pd.DataFrame, season_usage_: pd.DataFrame) -> pd.DataFrame:
    decisive = pitches_sit[pitches_sit['woba_value'].notna() & pitches_sit['pitch_type'].notna()].copy()
    decisive = decisive.merge(
        season_usage_[['pitcher', 'season', 'pitch_type', 'season_usage_rate']],
        on=['pitcher', 'season', 'pitch_type'], how='left',
    )
    for col in CATEGORICAL_FEATURES:
        decisive[col] = decisive[col].astype('category')
    return decisive


decisive_table = build_decisive_pitch_table(pitches, season_usage)
assert decisive_table['season_usage_rate'].notna().mean() > 0.99
print(f'결정구 {len(decisive_table):,}건, season_usage_rate 결측 {decisive_table["season_usage_rate"].isna().sum()}건')

결정구 917,717건, season_usage_rate 결측 0건


In [6]:
def fit_run_value_model(table: pd.DataFrame, seasons) -> HistGradientBoostingRegressor:
    train = table[table['season'].isin(seasons)]
    model = HistGradientBoostingRegressor(
        categorical_features='from_dtype', random_state=RANDOM_STATE, max_iter=200, early_stopping=True,
    )
    model.fit(train[MODEL_FEATURES], train['woba_value'])
    return model


train_set = decisive_table[decisive_table['season'].isin(TRAIN_SEASONS)]
valid_set = decisive_table[decisive_table['season'] == VALID_SEASON]
model_cv = fit_run_value_model(decisive_table, TRAIN_SEASONS)
pred_valid = model_cv.predict(valid_set[MODEL_FEATURES])
model_r2 = r2_score(valid_set['woba_value'], pred_valid)
print(f'(전체 피처) GBM 검증 R²: {model_r2:.4f}')
assert model_r2 > 0

final_model = fit_run_value_model(decisive_table, decisive_table['season'].unique())
print('(전체 피처) 최종 모델 준비 완료 -- 선택가치 계산에 씀')

(전체 피처) GBM 검증 R²: 0.0568


(전체 피처) 최종 모델 준비 완료 -- 선택가치 계산에 씀


In [7]:
def find_rematch_outcome_pitch(xbh_: pd.DataFrame, pitches_sit: pd.DataFrame) -> pd.DataFrame:
    decisive = pitches_sit[pitches_sit['woba_value'].notna()][
        ['game_pk', 'pitcher', 'season', 'at_bat_number', *MODEL_FEATURES, 'platoon_match', 'events', 'woba_value']
    ]
    keys = xbh_[['game_pk', 'pitcher', 'next_at_bat_number', 'reused_same_type']].rename(
        columns={'next_at_bat_number': 'at_bat_number'}
    )
    out = keys.merge(decisive, on=['game_pk', 'pitcher', 'at_bat_number'], how='left', validate='many_to_one')
    return out


REQUIRED_OUTCOME_COLUMNS = sorted(set(['events', 'woba_value', 'platoon_match', *MODEL_FEATURES]))


def build_avoidance_outcome_dataset(found: pd.DataFrame) -> pd.DataFrame:
    out = found.copy()
    out['pitch_family'] = out['pitch_type'].map(map_pitch_family)
    out = out.dropna(subset=[*REQUIRED_OUTCOME_COLUMNS, 'pitch_family', 'reused_same_type'])
    out['avoided'] = (1 - out['reused_same_type']).astype(int)
    out['allowed_xbh_again'] = out['events'].isin(EXTRA_BASE_HIT_EVENTS).astype(int)
    out['pitcher'] = out['pitcher'].astype(str)
    return out


found = find_rematch_outcome_pitch(xbh, decisive_table)
found_with_pa = found.dropna(subset=['woba_value'])
model_data = build_avoidance_outcome_dataset(found_with_pa)
assert len(model_data) == 27_245, len(model_data)
print(f'모델링에 쓸 재대결: {len(model_data):,}건, 투수 {model_data["pitcher"].nunique()}명 (노트북 07과 동일해야 함)')

모델링에 쓸 재대결: 27,245건, 투수 695명 (노트북 07과 동일해야 함)


## 4. 반사실(counterfactual) 구종 준비

"맞은 구종을 재사용했다고 가정"하는 비교를 하려면 각 재대결 이벤트에 그 정보가 필요합니다. 각 재대결
이벤트에 **맞은 구종**(`hit_pitch_type`, 원래 장타를 맞았던 구종)과, 그 투수·시즌에서 그 구종을 쓴
비율(`hit_pitch_season_usage_rate`, GBM의 `season_usage_rate` 피처를 채우는 데 필요)을 붙입니다. 그
시즌에 한 번도 안 던진 구종이면 사용률은 0으로 채웁니다(결측이 아니라 "그 시즌엔 안 씀"이 맞는 값).

In [8]:
def attach_counterfactual_pitch_type(model_data_: pd.DataFrame, xbh_: pd.DataFrame, season_usage_: pd.DataFrame) -> pd.DataFrame:
    cf_keys = xbh_[['game_pk', 'pitcher', 'next_at_bat_number', 'hit_pitch_type']].rename(
        columns={'next_at_bat_number': 'at_bat_number'}
    )
    cf_keys['pitcher'] = cf_keys['pitcher'].astype(str)
    out = model_data_.merge(cf_keys, on=['game_pk', 'pitcher', 'at_bat_number'], how='left', validate='many_to_one')

    usage = season_usage_.rename(
        columns={'pitch_type': 'hit_pitch_type', 'season_usage_rate': 'hit_pitch_season_usage_rate'}
    ).copy()
    usage['pitcher'] = usage['pitcher'].astype(str)
    out = out.merge(
        usage[['pitcher', 'season', 'hit_pitch_type', 'hit_pitch_season_usage_rate']],
        on=['pitcher', 'season', 'hit_pitch_type'], how='left',
    )
    out['hit_pitch_season_usage_rate'] = out['hit_pitch_season_usage_rate'].fillna(0.0)
    return out

In [9]:
_md_fix = pd.DataFrame({
    'game_pk': [100, 100], 'pitcher': ['10', '10'], 'at_bat_number': [5, 9], 'season': [2021, 2021],
})
_xbh_fix = pd.DataFrame({
    'game_pk': [100, 100], 'pitcher': [10, 10], 'next_at_bat_number': [5.0, 9.0],
    'hit_pitch_type': ['SL', 'CH'],
})
_su_fix = pd.DataFrame({'pitcher': [10], 'season': [2021], 'pitch_type': ['SL'], 'season_usage_rate': [0.35]})
_cf = attach_counterfactual_pitch_type(_md_fix, _xbh_fix, _su_fix)
assert _cf['hit_pitch_type'].tolist() == ['SL', 'CH']
assert np.isclose(_cf.loc[_cf['at_bat_number'] == 5, 'hit_pitch_season_usage_rate'].iloc[0], 0.35)
assert _cf.loc[_cf['at_bat_number'] == 9, 'hit_pitch_season_usage_rate'].iloc[0] == 0.0
print('attach_counterfactual_pitch_type 테스트 통과')

attach_counterfactual_pitch_type 테스트 통과


In [10]:
model_data = attach_counterfactual_pitch_type(model_data, xbh, season_usage)
assert model_data['hit_pitch_type'].notna().all()
print('반사실 구종 결측 0건 확인')

반사실 구종 결측 0건 확인


## 5. 선택가치 계산 (결정구 기준)

2절에서 정의한 `선택가치 = 예측(재사용 가정) − 예측(실제 선택)`을 각 재대결의 **결정구(타석 마지막
투구)**에 대해 계산합니다. 검산 삼아, 결정구가 우연히 맞은 구종과 같은 행(= 그 투구만 보면 "재사용"인
경우)의 선택가치는 반사실과 실제가 같은 구종이므로 정확히 0이 나와야 합니다.

In [11]:
def compute_selection_value(df: pd.DataFrame, model) -> pd.Series:
    actual_pred = model.predict(df[MODEL_FEATURES])
    cf = df[MODEL_FEATURES].copy()
    cf['pitch_type'] = df['hit_pitch_type'].astype(df['pitch_type'].dtype)
    cf['season_usage_rate'] = df['hit_pitch_season_usage_rate']
    cf_pred = model.predict(cf)
    return pd.Series(cf_pred - actual_pred, index=df.index)

In [12]:
class _LinearStub:
    def predict(self, X):
        return np.where(X['pitch_type'].astype(str) == 'SL', 0.5, 0.2)


_cat_dtype = pd.CategoricalDtype(categories=['SL', 'FF', 'CH'])
_df_sv = pd.DataFrame({
    'pitch_type': pd.Categorical(['FF', 'SL'], dtype=_cat_dtype), 'hit_pitch_type': ['SL', 'SL'],
    'season_usage_rate': [0.4, 0.5], 'hit_pitch_season_usage_rate': [0.3, 0.5],
    'stand': ['R', 'R'], 'p_throws': ['R', 'R'], 'balls': [0, 0], 'strikes': [0, 0],
    'outs_when_up': [0, 0], 'runners_n': [0, 0], 'risp': [0, 0], 'score_diff': [0, 0],
})
_sv = compute_selection_value(_df_sv, _LinearStub())
assert np.allclose(_sv.to_numpy(), [0.3, 0.0])
print('compute_selection_value 테스트 통과')

compute_selection_value 테스트 통과


In [13]:
model_data['selection_value_decisive_only'] = compute_selection_value(model_data, final_model)
_same_type_mask = model_data['pitch_type'].astype(str) == model_data['hit_pitch_type'].astype(str)
print('검산 -- 결정구가 맞은 구종과 같은 행(재사용)의 선택가치 평균(0이어야 함):',
      model_data.loc[_same_type_mask, 'selection_value_decisive_only'].mean())
display(model_data.groupby('avoided')['selection_value_decisive_only'].describe().round(4))

검산 -- 결정구가 맞은 구종과 같은 행(재사용)의 선택가치 평균(0이어야 함): 0.0


,count,mean,std,min,25%,50%,75%,max
avoided,,,,,,,,
0,12440.0,0.0015,0.0182,-0.1904,0.0000,0.0000,0.0070,0.1514
1,14805.0,0.0040,0.0308,-0.3122,-0.0137,0.0041,0.0215,0.3239


## 6. 처치·결과 단위 맞추기: 타석 전체로 집계

여기서 짚어야 할 설계 문제가 하나 있습니다. 위 `selection_value_decisive_only`는 재대결 타석의
**결정구(마지막 투구) 하나만** 보고 계산했는데, 우리가 효과를 보려는 `avoided`는 **타석 전체**에서
맞은 구종을 한 번도 안 던졌는지를 봅니다. 그래서 `avoided=0`(타석 중 어딘가에서는 재사용함) 타석의
결정구가 하필 맞은 구종이 아닐 수도 있고, 반대로 `avoided=1`인데 결정구 하나만 평가하는 식으로, 두
변수가 같은 단위를 보고 있지 않았습니다.

이걸 고치기 위해 재대결 타석에 실제로 던진 **모든 투구**에 대해 선택가치를 각각 계산하고(각 투구
자신의 볼카운트/상황을 그대로 사용), 타석 단위로 평균을 내 `avoided`와 같은 단위로 맞춥니다.
`avoided=1` 타석은 정의상 모든 투구가 맞은 구종이 아니므로 매 투구가 유효한 비교지만, `avoided=0`
타석은 재사용한 투구마다 선택가치가 정확히 0이 되어 평균을 0 쪽으로 끌어내립니다 — 이건 오류가 아니라
"부분적으로만 회피한 타석"을 올바르게 반영하는 것입니다.

In [14]:
def build_rematch_pa_pitches(xbh_: pd.DataFrame, pitches_sit: pd.DataFrame, season_usage_: pd.DataFrame, pitch_type_dtype) -> pd.DataFrame:
    keys = xbh_[['game_pk', 'pitcher', 'season', 'next_at_bat_number', 'hit_pitch_type']].rename(
        columns={'next_at_bat_number': 'at_bat_number'}
    )
    cols = ['game_pk', 'pitcher', 'at_bat_number', 'pitch_type', 'stand', 'p_throws',
            'balls', 'strikes', 'outs_when_up', 'runners_n', 'risp', 'score_diff']
    out = keys.merge(pitches_sit[cols], on=['game_pk', 'pitcher', 'at_bat_number'], how='left')
    out = out.dropna(subset=['pitch_type'])

    out = out.merge(
        season_usage_[['pitcher', 'season', 'pitch_type', 'season_usage_rate']],
        on=['pitcher', 'season', 'pitch_type'], how='left',
    )
    usage = season_usage_.rename(
        columns={'pitch_type': 'hit_pitch_type', 'season_usage_rate': 'hit_pitch_season_usage_rate'}
    )
    out = out.merge(
        usage[['pitcher', 'season', 'hit_pitch_type', 'hit_pitch_season_usage_rate']],
        on=['pitcher', 'season', 'hit_pitch_type'], how='left',
    )
    out['hit_pitch_season_usage_rate'] = out['hit_pitch_season_usage_rate'].fillna(0.0)
    out = out.dropna(subset=['season_usage_rate'])
    out['pitch_type'] = out['pitch_type'].astype(pitch_type_dtype)
    out['pitcher'] = out['pitcher'].astype(str)
    for col in ('stand', 'p_throws'):
        out[col] = out[col].astype('category')
    return out


def aggregate_pa_selection_value(pa_pitches: pd.DataFrame, model) -> pd.Series:
    sv = compute_selection_value(pa_pitches, model)
    return pa_pitches.assign(_sv=sv).groupby(['game_pk', 'pitcher', 'at_bat_number'])['_sv'].mean()

In [15]:
_TRAINED_DTYPE = pd.CategoricalDtype(categories=['SL', 'FF', 'CU', 'CH'])

_xbh_pa = pd.DataFrame({
    'game_pk': [100], 'pitcher': [10], 'season': [2021], 'next_at_bat_number': [5.0], 'hit_pitch_type': ['SL'],
})
_px_pa = pd.DataFrame({
    'game_pk': [100, 100, 100, 100], 'pitcher': [10, 10, 10, 10], 'at_bat_number': [5, 5, 5, 9],
    'pitch_type': ['FF', 'CU', 'FF', 'SL'], 'stand': ['R'] * 4, 'p_throws': ['R'] * 4,
    'balls': [0, 1, 2, 0], 'strikes': [0, 1, 2, 0], 'outs_when_up': [0, 0, 0, 0],
    'runners_n': [0, 0, 0, 0], 'risp': [0, 0, 0, 0], 'score_diff': [0, 0, 0, 0],
})
_su_pa = pd.DataFrame({
    'pitcher': [10, 10, 10], 'season': [2021, 2021, 2021], 'pitch_type': ['FF', 'CU', 'SL'],
    'season_usage_rate': [0.5, 0.3, 0.2],
})
_pa_pitches = build_rematch_pa_pitches(_xbh_pa, _px_pa, _su_pa, _TRAINED_DTYPE)
assert len(_pa_pitches) == 3  # the at_bat_number=9 row belongs to a different PA, must be excluded
assert _pa_pitches['pitch_type'].astype(str).tolist() == ['FF', 'CU', 'FF']
print('build_rematch_pa_pitches 테스트 통과')

_pa_full_avoid = pd.DataFrame({
    'game_pk': [100] * 3, 'pitcher': ['10'] * 3, 'at_bat_number': [5] * 3,
    'pitch_type': pd.Categorical(['FF', 'CU', 'FF'], dtype=_TRAINED_DTYPE), 'hit_pitch_type': ['SL'] * 3,
    'season_usage_rate': [0.5, 0.3, 0.5], 'hit_pitch_season_usage_rate': [0.2, 0.2, 0.2],
    'stand': ['R'] * 3, 'p_throws': ['R'] * 3, 'balls': [0, 1, 2], 'strikes': [0, 1, 2],
    'outs_when_up': [0, 0, 0], 'runners_n': [0, 0, 0], 'risp': [0, 0, 0], 'score_diff': [0, 0, 0],
})
_out_a = aggregate_pa_selection_value(_pa_full_avoid, _LinearStub())
assert np.isclose(_out_a.loc[(100, '10', 5)], 0.3)

_pa_partial = pd.DataFrame({
    'game_pk': [200] * 3, 'pitcher': ['20'] * 3, 'at_bat_number': [7] * 3,
    'pitch_type': pd.Categorical(['FF', 'SL', 'CH'], dtype=_TRAINED_DTYPE), 'hit_pitch_type': ['SL'] * 3,
    'season_usage_rate': [0.5, 0.2, 0.3], 'hit_pitch_season_usage_rate': [0.2, 0.2, 0.2],
    'stand': ['R'] * 3, 'p_throws': ['R'] * 3, 'balls': [0, 1, 2], 'strikes': [0, 1, 2],
    'outs_when_up': [0, 0, 0], 'runners_n': [0, 0, 0], 'risp': [0, 0, 0], 'score_diff': [0, 0, 0],
})
_out_b = aggregate_pa_selection_value(_pa_partial, _LinearStub())
assert np.isclose(_out_b.loc[(200, '20', 7)], 0.2)  # one of 3 pitches trivially 0 -> dilutes the mean
print('aggregate_pa_selection_value 테스트 통과')

build_rematch_pa_pitches 테스트 통과
aggregate_pa_selection_value 테스트 통과


In [16]:
pa_pitches = build_rematch_pa_pitches(xbh, pitches, season_usage, decisive_table['pitch_type'].dtype)
pa_mean = aggregate_pa_selection_value(pa_pitches, final_model).rename('selection_value_pa_mean').reset_index()
model_data = model_data.merge(pa_mean, on=['game_pk', 'pitcher', 'at_bat_number'], how='left', validate='one_to_one')
assert model_data['selection_value_pa_mean'].notna().all(), '모든 재대결 타석에 최소 1개 투구는 있어야 함'

# 검산 -- avoided=1 타석은 정의상 '재사용(=0)'인 투구가 하나도 없어야 함
_avoided1_pitches = pa_pitches.merge(
    model_data.loc[model_data['avoided'] == 1, ['game_pk', 'pitcher', 'at_bat_number']],
    on=['game_pk', 'pitcher', 'at_bat_number'],
)
_trivial_in_avoided1 = (_avoided1_pitches['pitch_type'].astype(str) == _avoided1_pitches['hit_pitch_type'].astype(str)).sum()
print(f'검산 -- avoided=1 타석 안에 맞은 구종과 같은 투구가 있으면 0이어야 함: {_trivial_in_avoided1}')
assert _trivial_in_avoided1 == 0

display(model_data.groupby('avoided')['selection_value_pa_mean'].describe().round(4))
print('기존(결정구만) vs 수정(타석 평균) 상관:',
      model_data[['selection_value_decisive_only', 'selection_value_pa_mean']].corr().iloc[0, 1].round(4))

검산 -- avoided=1 타석 안에 맞은 구종과 같은 투구가 있으면 0이어야 함: 0


,count,mean,std,min,25%,50%,75%,max
avoided,,,,,,,,
0,12440.0,0.0004,0.0147,-0.1685,-0.0071,0.000,0.0092,0.0860
1,14805.0,0.0027,0.0289,-0.3122,-0.0132,0.003,0.0186,0.2898


기존(결정구만) vs 수정(타석 평균) 상관: 0.7411


## 7. 혼합효과 모델 적합

`selection_value_pa_mean`을 종속변수로, `avoided`를 핵심 독립변수로 둔 선형 혼합효과 모델(`lmer`)을
적합합니다. 투수마다 재대결 표본 수가 다르므로(최소 하한 없음) `(1 | pitcher)` 랜덤절편으로 투수 간
차이를 부분 풀링(partial pooling)합니다.

In [17]:
FORMULA_SELECTION = (
    'selection_value_pa_mean ~ avoided + outs_when_up + score_diff + platoon_match + risp + runners_n '
    '+ factor(season) + (1 | pitcher)'
)
fit_lmer(model_data, FORMULA_SELECTION)
print('=== 수렴 상태 ===')
print(check_convergence())
fixed_selection = extract_lmer_fixed_effects()
print('=== 고정효과 (avoided 포함) ===')
display(fixed_selection.round(4))
vc_selection = extract_variance_components('pitcher')
print('분산 성분:', vc_selection)
if vc_selection['pitcher_intercept'] is None or vc_selection['residual'] is None:
    icc_selection = None
    print('ICC 계산 불가')
else:
    icc_selection = compute_icc_gaussian(vc_selection['pitcher_intercept'], vc_selection['residual'])
    print(f'ICC: {icc_selection:.4f}')

2026-10-05 22:49:30,627 [WARNING] R callback write-console: Loading required package: Matrix
  


=== 수렴 상태 ===
OK: no convergence warnings; isSingular=False
=== 고정효과 (avoided 포함) ===


,term,estimate,std_error,t_value,ci_low,ci_high,p_value
0,(Intercept),-0.0000,0.0004,-0.0520,-0.0009,0.0008,0.9585
1,avoided,0.0023,0.0003,7.9111,0.0017,0.0028,0.0000
2,outs_when_up,-0.0002,0.0002,-1.0297,-0.0005,0.0002,0.3031
3,score_diff,0.0001,0.0000,1.3444,-0.0000,0.0002,0.1788
4,platoon_match,-0.0004,0.0003,-1.2708,-0.0009,0.0002,0.2038
5,risp,-0.0009,0.0005,-1.7939,-0.0020,0.0001,0.0728
6,runners_n,0.0009,0.0003,3.1164,0.0003,0.0015,0.0018
7,factor(season)2022,0.0003,0.0005,0.5498,-0.0007,0.0012,0.5824
8,factor(season)2023,0.0004,0.0005,0.8622,-0.0005,0.0014,0.3886
9,factor(season)2024,0.0007,0.0005,1.3955,-0.0003,0.0016,0.1629


분산 성분: {'pitcher_intercept': 1.5992607246188007e-06, 'residual': 0.0005512811104788295}
ICC: 0.0029


## 8. 견고성 검증 (B): 공변량이 진짜 '타석 시작 전' 상태인가

지금까지 회귀의 `outs_when_up`/`score_diff`/`runners_n`/`risp`/`platoon_match`는 결정구(타석 마지막
투구) 시점 값을 썼고, "타석 중엔 거의 안 바뀐다"고 가정만 했습니다(6절 참고 — 이들이 바뀌면 "회피"라는
타석 전체 수준 선택의 효과를 통제하면서 같이 지워버릴 위험이 있습니다). 실제로 그런지, 같은 타석의
**첫 투구** 시점 값과 비교해 직접 확인합니다 — 관측 안 되는 교란요인이 아니라 데이터 안에 이미 있는
값이라 바로 검증 가능합니다.

In [18]:
def build_pa_start_state(xbh_: pd.DataFrame, pitches_sit: pd.DataFrame) -> pd.DataFrame:
    keys = xbh_[['game_pk', 'pitcher', 'next_at_bat_number']].rename(columns={'next_at_bat_number': 'at_bat_number'})
    first_pitch = pitches_sit[pitches_sit['pitch_number'] == 1][
        ['game_pk', 'pitcher', 'at_bat_number', 'outs_when_up', 'score_diff', 'runners_n', 'risp', 'platoon_match']
    ]
    out = keys.merge(first_pitch, on=['game_pk', 'pitcher', 'at_bat_number'], how='left')
    return out

In [19]:
_xbh_start = pd.DataFrame({'game_pk': [100], 'pitcher': [10], 'next_at_bat_number': [5.0]})
_px_start = pd.DataFrame({
    'game_pk': [100, 100, 100], 'pitcher': [10, 10, 10], 'at_bat_number': [5, 5, 5], 'pitch_number': [1, 2, 3],
    'outs_when_up': [1, 1, 1], 'score_diff': [0, 0, -1], 'runners_n': [0, 1, 1], 'risp': [0, 0, 0],
    'platoon_match': [1.0, 1.0, 1.0],
})
_start = build_pa_start_state(_xbh_start, _px_start)
assert len(_start) == 1
assert _start.iloc[0]['score_diff'] == 0 and _start.iloc[0]['runners_n'] == 0  # first pitch's state, not later ones
print('build_pa_start_state 테스트 통과')

build_pa_start_state 테스트 통과


In [20]:
pa_start_state = build_pa_start_state(xbh, pitches)
pa_start_state['pitcher'] = pa_start_state['pitcher'].astype(str)
_compare = model_data[['game_pk', 'pitcher', 'at_bat_number', 'outs_when_up', 'score_diff', 'runners_n', 'risp', 'platoon_match']].merge(
    pa_start_state, on=['game_pk', 'pitcher', 'at_bat_number'], how='left', suffixes=('_decisive', '_start'),
)
print(f'타석 시작 상태를 못 찾은 재대결: {_compare["outs_when_up_start"].isna().sum()}건 / {len(_compare):,}건')
_compare = _compare.dropna(subset=['outs_when_up_start'])
for col in ['outs_when_up', 'score_diff', 'runners_n', 'risp', 'platoon_match']:
    mismatch = (_compare[f'{col}_decisive'] != _compare[f'{col}_start']).mean()
    print(f'  {col}: 결정구 시점 ≠ 타석 시작 시점 비율 = {mismatch:.2%}')

타석 시작 상태를 못 찾은 재대결: 0건 / 27,245건
  outs_when_up: 결정구 시점 ≠ 타석 시작 시점 비율 = 0.30%
  score_diff: 결정구 시점 ≠ 타석 시작 시점 비율 = 0.17%
  runners_n: 결정구 시점 ≠ 타석 시작 시점 비율 = 0.46%
  risp: 결정구 시점 ≠ 타석 시작 시점 비율 = 1.73%
  platoon_match: 결정구 시점 ≠ 타석 시작 시점 비율 = 0.01%


## 9. 견고성 검증 (C): GBM 재적합 불확실성

7절의 `lmer`가 보고하는 표준오차는 `selection_value_pa_mean`을 고정된 값처럼 취급합니다. 하지만 그
값 자체가 GBM의 예측값이라, GBM을 다른 표본으로 다시 학습했다면 값이 조금씩 달랐을 겁니다. 결정구
917,717건을 복원추출로 재표본하고 GBM을 B번 다시 학습해서, "GBM을 다시 학습했다면 `avoided` 효과
(평균 차이)가 얼마나 달라졌을지"의 표준편차를 직접 구해 `lmer`의 SE와 비교합니다.

In [21]:
def gbm_refit_bootstrap_gap_sd(decisive_table_, pa_pitches_, avoided_by_pa, seasons, n_boot, random_state):
    rng = np.random.default_rng(random_state)
    train = decisive_table_[decisive_table_['season'].isin(seasons)]
    gaps = []
    for b in range(n_boot):
        resample_idx = rng.integers(0, len(train), size=len(train))
        boot_train = train.iloc[resample_idx]
        model = HistGradientBoostingRegressor(
            categorical_features='from_dtype', random_state=random_state + b, max_iter=200, early_stopping=True,
        )
        model.fit(boot_train[MODEL_FEATURES], boot_train['woba_value'])
        pa_mean = aggregate_pa_selection_value(pa_pitches_, model).reindex(avoided_by_pa.index)
        gaps.append(float(pa_mean[avoided_by_pa == 1].mean() - pa_mean[avoided_by_pa == 0].mean()))
    return gaps

In [22]:
_rng_t = np.random.default_rng(0)
_n_t = 40
_decisive_t = pd.DataFrame({
    'season': [2021] * _n_t,
    'pitch_type': pd.Categorical(_rng_t.choice(['FF', 'SL'], size=_n_t), categories=['FF', 'SL']),
    'stand': pd.Categorical(['R'] * _n_t), 'p_throws': pd.Categorical(['R'] * _n_t),
    'balls': _rng_t.integers(0, 4, _n_t), 'strikes': _rng_t.integers(0, 3, _n_t),
    'outs_when_up': _rng_t.integers(0, 3, _n_t), 'runners_n': _rng_t.integers(0, 4, _n_t),
    'risp': _rng_t.integers(0, 2, _n_t), 'score_diff': _rng_t.integers(-3, 3, _n_t),
    'season_usage_rate': _rng_t.uniform(0.1, 0.9, _n_t), 'woba_value': _rng_t.uniform(0, 1, _n_t),
})
_pa_t = pd.DataFrame({
    'game_pk': [1, 1, 2, 2], 'pitcher': ['1', '1', '2', '2'], 'at_bat_number': [5, 5, 7, 7],
    'pitch_type': pd.Categorical(['FF', 'SL', 'FF', 'FF'], categories=['FF', 'SL']),
    'hit_pitch_type': ['SL', 'SL', 'FF', 'FF'],
    'season_usage_rate': [0.5, 0.5, 0.5, 0.5], 'hit_pitch_season_usage_rate': [0.3, 0.3, 0.2, 0.2],
    'stand': pd.Categorical(['R'] * 4), 'p_throws': pd.Categorical(['R'] * 4),
    'balls': [0, 1, 0, 1], 'strikes': [0, 1, 0, 1], 'outs_when_up': [0, 0, 1, 1],
    'runners_n': [0, 0, 0, 0], 'risp': [0, 0, 0, 0], 'score_diff': [0, 0, 0, 0],
})
_avoided_t = pd.Series({(1, '1', 5): 1, (2, '2', 7): 0})
_avoided_t.index = pd.MultiIndex.from_tuples(_avoided_t.index, names=['game_pk', 'pitcher', 'at_bat_number'])
_gaps_t = gbm_refit_bootstrap_gap_sd(_decisive_t, _pa_t, _avoided_t, [2021], n_boot=3, random_state=0)
assert len(_gaps_t) == 3 and all(np.isfinite(g) for g in _gaps_t)
print('gbm_refit_bootstrap_gap_sd 테스트 통과')

gbm_refit_bootstrap_gap_sd 테스트 통과


In [23]:
_avoided_by_pa = model_data.set_index(['game_pk', 'pitcher', 'at_bat_number'])['avoided']
N_BOOT = 50
boot_gaps = gbm_refit_bootstrap_gap_sd(
    decisive_table, pa_pitches, _avoided_by_pa, decisive_table['season'].unique(), N_BOOT, RANDOM_STATE,
)
boot_sd = float(np.std(boot_gaps))
_lmer_se = float(fixed_selection.loc[fixed_selection['term'] == 'avoided', 'std_error'].iloc[0])
_combined_se = float(np.sqrt(_lmer_se ** 2 + boot_sd ** 2))
print(f'GBM 재적합 부트스트랩 {N_BOOT}회: avoided 효과(평균 차이)의 표준편차 = {boot_sd:.5f}')
print(f'lmer가 보고한 SE = {_lmer_se:.5f}')
print(f'두 불확실성을 합친(제곱합 제곱근) 보수적 SE = {_combined_se:.5f}')

GBM 재적합 부트스트랩 50회: avoided 효과(평균 차이)의 표준편차 = 0.00067
lmer가 보고한 SE = 0.00029
두 불확실성을 합친(제곱합 제곱근) 보수적 SE = 0.00073


## 10. 견고성 검증 (D): 순열검정

`selection_value_pa_mean`은 재사용 투구가 섞이면 0에 질량이 쏠리는 분포라 가우시안 `lmer`의 정규성
가정과 맞지 않을 수 있습니다. `avoided`의 raw 평균 차이(공변량 보정 전, 6절 describe()의 두 그룹 평균
차이와 거의 같음)를 분포 가정 없이 순열검정으로 다시 확인합니다. ICC가 0.003 수준으로 작아 투수 군집을
무시한 전체 표본 셔플로도 충분하다고 보고, `avoided` 라벨을 무작위로 재배치해 귀무분포를 만듭니다.

In [24]:
def permutation_test_avoided_effect(values, avoided, n_perm, seed):
    rng = np.random.default_rng(seed)
    avoided = np.asarray(avoided)
    values = np.asarray(values, dtype=float)
    observed = float(values[avoided == 1].mean() - values[avoided == 0].mean())
    n = len(values)
    n1 = int(avoided.sum())
    null_gaps = np.empty(n_perm)
    for i in range(n_perm):
        perm_idx = rng.permutation(n)
        chosen, rest = perm_idx[:n1], perm_idx[n1:]
        null_gaps[i] = values[chosen].mean() - values[rest].mean()
    p_value = float((np.abs(null_gaps) >= abs(observed)).mean())
    return {'observed_gap': observed, 'p_value': p_value, 'null_sd': float(null_gaps.std())}

In [25]:
_v = np.array([0.0] * 5 + [1.0] * 5)
_a = np.array([0] * 5 + [1] * 5)
_res = permutation_test_avoided_effect(_v, _a, n_perm=500, seed=1)
assert np.isclose(_res['observed_gap'], 1.0)
assert _res['p_value'] < 0.05  # perfectly separated groups should sit in the extreme tail of the null
print('permutation_test_avoided_effect 테스트 통과')

permutation_test_avoided_effect 테스트 통과


In [26]:
_perm_result = permutation_test_avoided_effect(
    model_data['selection_value_pa_mean'].to_numpy(), model_data['avoided'].to_numpy(), n_perm=5000, seed=RANDOM_STATE,
)
print(f'관측된 평균 차이: {_perm_result["observed_gap"]:.5f}')
print(f'순열검정 p-value (5000회): {_perm_result["p_value"]:.5f}')
print(f'귀무분포 표준편차: {_perm_result["null_sd"]:.5f}')

관측된 평균 차이: 0.00234
순열검정 p-value (5000회): 0.00000
귀무분포 표준편차: 0.00028


## 11. 효과 크기를 실전 단위로 환산: "유의함"과 "중요함"은 다르다

지금까지의 검증으로 `avoided` 효과는 통계적으로 0과 구별된다는 게 확인됐습니다. 하지만 그 크기
(+0.0023, `woba_value`와 같은 단위)가 실전에서 어느 정도인지는 감이 잘 오지 않습니다. wOBA 포인트를
득점가치(run)로 환산하는 표준 계수(wOBA scale, 대략 1.2 — 연도별로 1.15~1.25 사이)로 나누고, 이
표본에서 투수 한 명이 실제로 겪는 재대결 빈도를 곱해서, "이 선택을 항상 완벽하게 했다면 투수 한 명에게
실제로 얼마나 이득이었을지"를 추정합니다.

In [27]:
WOBA_SCALE = 1.2  # wOBA 포인트 -> 득점가치(run) 환산 계수의 대표값

selection_effect_woba = float(fixed_selection.loc[fixed_selection['term'] == 'avoided', 'estimate'].iloc[0])
runs_per_decision = selection_effect_woba / WOBA_SCALE

rematches_per_pitcher = model_data.groupby('pitcher').size()
avg_rematches_per_pitcher = float(rematches_per_pitcher.mean())
n_seasons_in_sample = int(pitches['season'].nunique())

total_runs_over_sample = runs_per_decision * avg_rematches_per_pitcher
runs_per_pitcher_per_season = total_runs_over_sample / n_seasons_in_sample

print(f'재대결 1건당 선택가치 효과: {selection_effect_woba:.4f} wOBA ≈ {runs_per_decision:.5f} 득점')
print(f'투수 1명당 평균 재대결 수(표본 {n_seasons_in_sample}개 시즌 누적): {avg_rematches_per_pitcher:.1f}건')
print(f'→ 이 선택을 항상 완벽히 했다면, 투수 1명이 표본 전체 {n_seasons_in_sample}개 시즌 동안 얻는 총 이득: '
      f'약 {total_runs_over_sample:.3f}득점')
print(f'→ 시즌당으로 나누면: 약 {runs_per_pitcher_per_season:.4f}득점/시즌')
print('(참고: 투수 1승(win)은 대략 10득점 안팎 -- 이 효과는 선수 가치 평가에 쓰이는 어떤 단위로도 '
      '거의 감지되지 않는 크기입니다)')

재대결 1건당 선택가치 효과: 0.0023 wOBA ≈ 0.00189 득점
투수 1명당 평균 재대결 수(표본 6개 시즌 누적): 39.2건
→ 이 선택을 항상 완벽히 했다면, 투수 1명이 표본 전체 6개 시즌 동안 얻는 총 이득: 약 0.074득점
→ 시즌당으로 나누면: 약 0.0124득점/시즌
(참고: 투수 1승(win)은 대략 10득점 안팎 -- 이 효과는 선수 가치 평가에 쓰이는 어떤 단위로도 거의 감지되지 않는 크기입니다)


## 12. 종합 해석과 한계

* **처치·결과 단위를 맞춘 결과.** `avoided`(타석 전체 기준)와 같은 단위로 집계한
  `selection_value_pa_mean`으로 적합한 혼합효과 모델에서, `avoided` 추정치는 **+0.0023**
  [95% CI 0.0017, 0.0028], **p < 0.0001** (t=7.91) 입니다. 결정구 하나만 보고 계산한 버전(+0.0025,
  5절)과 거의 같은 크기·방향·유의성이라(상관 0.7411), 6절에서 고친 단위 불일치가 결론을 바꾸지는
  않았습니다.
* **견고성 검증 (B) — 통과.** 결정구 시점 공변량이 타석 시작 시점과 다른 비율은 `outs_when_up` 0.30%,
  `score_diff` 0.17%, `runners_n` 0.46%, `risp` 1.73%, `platoon_match` 0.01%로 전부 2% 미만입니다.
  도루·보크 등으로 생기는 드문 변화일 뿐, "타석 중엔 거의 안 바뀐다"는 가정은 실측으로 뒷받침됩니다.
* **견고성 검증 (D) — 통과.** 5,000회 순열검정에서 관측된 평균 차이(+0.00234)보다 극단적인 경우가
  0건이라 p≈0 (<0.0002)이고, 귀무분포 표준편차(0.00028)는 `lmer`가 보고한 SE(0.00029)와 거의
  같습니다. 분포 가정 없는 방법과 `lmer`가 거의 같은 답을 내놓았으니, `selection_value`의 0 근처 질량
  쏠림이 결론을 왜곡하지는 않았습니다.
* **견고성 검증 (C) — 중요한 발견, 결론이 약화됩니다.** GBM을 50번 다시 학습했을 때 `avoided` 효과
  (평균 차이)의 표준편차는 **0.00067**로, `lmer`가 보고한 SE(0.00029)보다 **2배 이상 큽니다** —
  원래 보고된 불확실성보다 GBM 재학습 자체의 변동성이 더 큰 불확실성 원천이었습니다. 두 불확실성을
  제곱합 제곱근으로 합친 보수적 SE는 **0.00073**이고, 이 기준으로 다시 계산하면 t=3.15, **p≈0.0016**,
  95% CI는 **[0.00087, 0.00373]**입니다. 0을 여전히 벗어나 있어 효과가 사라지지는 않지만, 원래
  "p<0.0001, 매우 정밀하게 검출"이라던 확신은 과장이었습니다.
* **유의함 ≠ 중요함.** 11절에서 본 대로, 이 효과를 실전 단위로 환산하면 투수 1명이 표본 전체 기간
  (2021~2026, 6개 시즌) 동안 얻는 총 이득이 1득점에도 한참 못 미칩니다(1승 ≈ 10득점). n=27,245라는
  큰 표본이 주는 통계적 검정력 덕분에 "정확히 0은 아니다"는 잡아내지만, 이게 "회피가 중요한 선수
  가치 요소"라는 뜻은 전혀 아닙니다.
* **종합 결론.** 회피가 아예 무의미한 선택은 아닙니다 — GBM이 학습한 상황-구종 관계를 기준으로, 맞은
  구종을 피하는 선택은 재사용하는 것보다 통계적으로 구별 가능한 수준으로 더 낮은 기대 피해를
  보였습니다. 하지만 그 크기는 실전에서 체감되거나 선수 가치 평가에 반영될 수준이 아닙니다. "회피가
  유리하다"를 이 프로젝트의 실질적 발견으로 내세우기엔 근거가 약합니다.
* **여전히 남아 있는 한계.** 이번 노트북은 처치·결과 단위 불일치를 고치고, 공변량 가정과 분포 가정
  문제를 실측으로 검증했고, 1단계 모델 불확실성을 반영해 유의성을 재평가했습니다. 하지만 다음은
  손대지 못했습니다.
  - **선택편향.** 그날 컨디션이 좋은 투수가 회피도 하고 다른 구종도 잘 던졌을 수 있다는, 관측되지
    않는 공통원인 문제입니다. GBM에 `pitcher_id`를 넣으면 줄어들 수 있지만, 그러면 반사실 swap이
    "이 투수가 거의/전혀 안 던지는 구종을 던졌다면"을 예측해야 하는 칸이 극도로 희박해져 외삽 위험이
    오히려 커집니다 — 선택편향을 줄이려는 시도가 외삽 위험을 키우는 구조적 트레이드오프라, 지금
    설계로는 둘 다 해결할 방법이 없습니다.
  - **GBM의 `pitch_type` 효과가 매치업이 아니라 리그 평균 수준.** 타자 식별 정보가 피처에 없어서,
    "이 특정 타자를 상대로 영리하게 피했다"가 아니라 "평균적으로 덜 위험한 구종으로 바꿨다"는
    증거로 읽어야 합니다.
  - **다중비교 미보정.** 이 프로젝트에서 RVAE → 혼합효과 모델(D/E) → 선택가치까지 여러 지표를
    시도했고 그중 선택가치만 유의했습니다. 사전에 이 지표 하나만 검증하기로 정하고 돌린 게 아니라
    여러 개를 시도하다 걸린 것이라, 다중비교 보정 없이 "유의함"을 강조하면 과장입니다.
  - **`season_usage_rate`가 시즌 전체 집계라 미래 정보를 포함.** 노트북 03 때부터 있던 설계고,
    이번에 건드리지 않았습니다.